Created by Muhid Qaiser 

Email : muhidqaiser02@gmail.com 

Linkedin : https://www.linkedin.com/in/muhid-qaiser/

Github : https://github.com/Muhid-Qaiser

# The Neuron Gorge

https://sites.google.com/view/theneurongorge

# Simulated Annealing Search

#### Table of Contents

- Introduction
- Simple Implementations
    - Global Optimization using Simulated Annealing 
    - Budget-Constrained Feature Selection using Simulated Annealing
- Practice Questions


## Introduction

### Simulated Annealing

**Simulated Annealing** is a randomized optimization method that mimics cooling metal: you start at a high “temperature” to freely explore solutions (even accepting worse ones), then gradually lower it to focus on improvements. At each step you tweak the current solution, compute the change in “energy” (objective), and accept it if it’s better, or with probability exp(–ΔE/T) if it’s worse, letting you escape local minima and converge on a good solution.

### Simulated-Annealing vs Hill-Climbing

**Hill Climbing** and **Simulated Annealing** both iteratively improve a solution, but:

- **Move Acceptance**  
  - *Hill Climbing* only accepts moves that improve (lower) the objective, so it’s purely greedy.  
  - *Simulated Annealing* accepts worse moves with probability \(\exp(-\Delta E/T)\), allowing occasional “uphill” steps.

- **Escaping Local Optima**  
  - *Hill Climbing* gets stuck as soon as no neighbor is better.  
  - *Simulated Annealing* can jump out of local optima early on (when \(T\) is high), then gradually “freeze” into a good region.

- **Control Parameter**  
  - *Hill Climbing* has no temperature, its search is static.  
  - *Simulated Annealing* uses a **cooling schedule** (\(T\) decreases over time) to balance exploration vs. exploitation.

- **Stochastic vs. Deterministic**  
  - *Hill Climbing* is deterministic (given tie‑breaking), always moving uphill until plateau.  
  - *Simulated Annealing* is stochastic throughout, with randomness controlled by temperature.

In short, hill climbing is fast but can’t recover from local traps; simulated annealing trades off extra randomness (and runtime) to explore more broadly and avoid premature convergence.

### Limitations

- **No optimality guarantee**: Only probabilistic, may miss the global best.  
- **Parameter tuning**: Initial temperature, cooling rate, and iterations need careful choice.  
- **Slow convergence**: Can require many evaluations to “cool” properly.  
- **Stochastic results**: Different runs can give different solutions.  
- **Neighbor design**: Effective moves often need problem‑specific crafting.



## Simple Implementations

### Global Optimization using Simulated Annealing

In [6]:
import math
import random

def acceptance_probability(delta_E, temperature):
    if temperature <= 0:
        return 0.0  # * never accept if T is zero or negative
    if delta_E < 0:
        return 1.0
    return math.exp(-delta_E / temperature)

def simulated_annealing(energy_values, initial_temperature, cooling_rate, min_temperature=1e-6):
    # * Start from a random state
    current_idx = random.randrange(len(energy_values))
    best_idx = current_idx
    T = initial_temperature

    while T > min_temperature:
        # * 1) Propose a random neighbor (uniformly)
        candidate_idx = random.randrange(len(energy_values))

        # * 2) Compute energy difference ΔE = E_candidate - E_current
        delta_E = energy_values[candidate_idx] - energy_values[current_idx]

        # * 3) Accept move if it's better, or with Metropolis probability
        if delta_E < 0 or random.random() < acceptance_probability(delta_E, T):
            current_idx = candidate_idx
            # * Update best if improved
            if energy_values[current_idx] < energy_values[best_idx]:
                best_idx = current_idx

        # * 4) Cool down
        T *= cooling_rate

    return best_idx

# * --------------------------
# * Example usage:
energy_values = [10.0,  9.8,  9.5,  9.7,  9.0, 10.2]  # * lower is better
T0 = 1.0
alpha = 0.9

best = simulated_annealing(energy_values, T0, alpha)
print("Best state index:", best)
print("Best energy:", energy_values[best])


Best state index: 4
Best energy: 9.0


### Budget-Constrained Feature Selection using Simulated Annealing

In [9]:
import math
import random

def total_cost(selection, states):
    # * Sum of costs for the selected features
    return sum(states[f]["cost"] for f in selection)

def total_rating(selection, states):
    # * Sum of ratings for the selected features
    return sum(states[f]["rating"] for f in selection)

def acceptance_probability(delta_E, T):
    # * Metropolis criterion: always accept if delta_E < 0 (energy decreases), 
    # * else accept with probability exp(-delta_E / T).
    if T <= 0:
        return 0.0
    if delta_E < 0:
        return 1.0
    return math.exp(-delta_E / T)

def random_neighbor(current, states, budget):
    
    # * Propose a neighbor by either adding or removing one feature:
    # *   - If adding: pick a feature not in current that fits within budget.
    # *   - If removing: pick one feature from current.
    # * If one move type is impossible, do the other.
    
    features = list(states.keys())
    current_cost = total_cost(current, states)
    
    # * * Decide move type
    can_add = any(
        f not in current and current_cost + states[f]["cost"] <= budget
        for f in features
    )
    can_remove = len(current) > 0
    
    # * * If neither move is possible (e.g. budget=0), return current
    if not can_add and not can_remove:
        return set(current)
    
    # * Randomly choose to add or remove (weighted if one move only)
    if can_add and (not can_remove or random.random() < 0.5):
        # * Add move
        candidates = [
            f for f in features
            if f not in current and current_cost + states[f]["cost"] <= budget
        ]
        f = random.choice(candidates)
        neighbor = set(current)
        neighbor.add(f)
        return neighbor
    else:
        # * Remove move
        f = random.choice(list(current))
        neighbor = set(current)
        neighbor.remove(f)
        return neighbor

def simulated_annealing_feature_selection( states, budget, initial_temperature=100.0, cooling_rate=0.95, max_iterations=1000):
    # *    Simulated Annealing for selecting a subset of 'states' under 'budget'
    # *    to maximize total_rating.
    # * Initialize with an empty (feasible) selection
    
    current = set()
    best = set()
    
    T = initial_temperature

    for iteration in range(max_iterations):
        # * 1) Propose a neighbor
        candidate = random_neighbor(current, states, budget)
        
        # * 2) Compute energies: E = -rating
        E_current = -total_rating(current, states)
        E_candidate = -total_rating(candidate, states)
        delta_E = E_candidate - E_current
        
        # * 3) Accept or reject
        if delta_E < 0 or random.random() < acceptance_probability(delta_E, T):
            current = candidate
            # * Update best if improved
            if total_rating(current, states) > total_rating(best, states):
                best = set(current)
        
        # * 4) Cool down
        T *= cooling_rate

    return best


# * Example usage
if __name__ == "__main__":
    budget = 750
    states = {
        "feature1": {"cost": 100, "rating": 3.5},
        "feature2": {"cost": 200, "rating": 4.2},
        "feature3": {"cost": 150, "rating": 4.0},
        "feature4": {"cost": 300, "rating": 3.8},
        "feature5": {"cost": 250, "rating": 4.5},
        "feature6": {"cost": 350, "rating": 3.6}
    }

    selected = simulated_annealing_feature_selection(
        states,
        budget,
        initial_temperature=100.0,
        cooling_rate=0.9,
        max_iterations=1000
    )

    print("Selected features:", selected)
    print("Total cost:", total_cost(selected, states))
    print("Total rating:", total_rating(selected, states))


Selected features: {'feature1', 'feature5', 'feature3', 'feature2'}
Total cost: 700
Total rating: 16.2


## Practice Question

### Q1 : 8-Puzzle Problem 
Given a 3×3 board with 8 numbered tiles (1-8) and a blank space, solve the puzzle by arranging the tiles in increasing order using the least number of moves.

Use Manhattan Distance as the heuristic.

Implement various Simulated Annealing search strategies and compare results.

Use the given Intial and Goal States

In [ ]:
initial_state = [
['1', '2', '3'],
['4', '5', '6'],
['7', '_', '8']
]

goal_state = [
['1', '2', '3'],
['_', '4', '6'],
['7', '5', '8']
]

In [ ]:
# * Code Here

### Q2: Job Scheduling Problem

You have **N jobs** and **M machines**. Each job has a processing time and must be assigned to a machine in such a way that the **total completion time** (the time at which the last job finishes) is minimized.

Implement **Simulated Annealing search**.

Additionally, handle **job dependencies** (e.g., Job J2 depends on Job J1, so J2 must start after J1 finishes).

---

#### **Input:**

- **Jobs**: { 'J1', 'J2', 'J3', 'J4' }

- **Processing Times**: 
    - J1 takes 5 units, J2 takes 10 units, J3 takes 3 units, J4 takes 7 units.

- **Machines**: { 'M1', 'M2' }

---

#### **Dependencies:**

- J2 depends on J1 (J2 must start after J1 finishes).
- J4 depends on J3 (J4 must start after J3 finishes).

---

#### **Objective:**

Minimize the total completion time, i.e., the time at which the last job finishes, while respecting the dependencies between jobs.

---

#### **Output:**

For each strategy, display the following:

1. **Job Assignments**: 
    - Example: 
      - M1: { J1, J3 }
      - M2: { J2, J4 }

2. **Machine Completion Times**: 
    - Example: 
      - M1: 8
      - M2: 17

3. **Total Completion Time**: 17


In [ ]:
# * Code Here

### Q3: Traveling Salesman Problem (TSP)

You are given a set of cities and the distances between them. The objective is to find the shortest possible route that visits each city exactly once and returns to the origin city.

**Input:**

Cities: { 'A', 'B', 'C', 'D', 'E' }

Distances (in arbitrary units):

Dist(A, B) = 10, Dist(A, C) = 15, Dist(A, D) = 20, Dist(A, E) = 25

Dist(B, C) = 35, Dist(B, D) = 30, Dist(B, E) = 50

Dist(C, D) = 15, Dist(C, E) = 40

Dist(D, E) = 10

**Objective:** Use Simulated Annealing to find a route with the minimal distance, ensuring that every city is visited once and returns to the origin city.

In [ ]:
# * Code Here

## Happy Coding :)